# 🧪 Predicting Photocatalytic H₂ Production Rate with Machine Learning

**A beginner-friendly walkthrough — no prior machine learning experience needed.**

This notebook builds a model that predicts how much hydrogen gas (H₂) a photocatalytic reaction will produce, based on the catalyst you use and the conditions you run the reaction under (pH, light source, catalyst loading, temperature, etc.). It's trained on 909 real experiments pulled from 119 published papers.

If you can read a calibration curve or interpret a DOE (Design of Experiments) table, you already have the background needed to follow this notebook — every machine learning concept is explained using an analogy to something in wet-lab chemistry.

---


## 📊 Results at a Glance

*(These are the actual results from running this exact notebook on the full 909-experiment dataset. You'll reproduce them yourself when you run the cells below — think of this section as the "Abstract" of the notebook.)*

### The headline number

We combined **three different models** into one "panel of experts" (a technique called *stacking*, explained in Section 7). That combined model explains **77.6% of the variation** in H₂ production rate across nearly a thousand experiments spanning 119 different labs' worth of catalysts and conditions — and it beats every individual model on its own.

| Model | R² (higher = better fit) | RMSE (typical error, μmol g⁻¹h⁻¹) | MAE (average error, μmol g⁻¹h⁻¹) |
|---|---|---|---|
| CatBoost (single model) | 0.757 | 12,713 | 4,525 |
| Random Forest (single model) | 0.602 | 16,274 | 5,547 |
| XGBoost (single model) | 0.651 | 15,237 | 5,307 |
| Simple average of the 3 | 0.671 | 14,808 | 5,032 |
| **🏆 Stacked ensemble (our final model)** | **0.776** | **12,208** | **4,337** |

*(Don't worry if R², RMSE, and MAE aren't familiar yet — Section 8 explains each one using a calibration-curve analogy you'll recognize.)*

### What actually drives H₂ yield? (from SHAP analysis, Section 10)

Beyond which paper/lab the data came from, the biggest drivers of predicted H₂ yield were, in order: **semiconductor identity** (Semiconductor 1 & 2), **how the photocatalyst was prepared**, its **crystal/support structure**, the **co-catalyst used**, and **catalyst loading (g/L)**. This matches basic photocatalysis intuition — catalyst composition and synthesis route are the first-order levers on activity — which is a good sanity check that the model learned real chemistry, not noise.

### The plots

Three result figures (`model_comparison.png`, `shap_summary_beeswarm.png`, `parity_plot.png`) are provided alongside this notebook as separate image files — open them directly to see the model comparison chart, the SHAP feature-impact summary, and the predicted-vs-actual parity plot. The same plots are also regenerated live in Sections 12–13 when you run the notebook end-to-end.

> 💡 **Why isn't R² closer to 1.0, like a good calibration curve?** A UV-Vis calibration curve is built from one instrument, one compound, and controlled standards — everything except concentration is held constant, so R² > 0.99 is expected. This dataset instead combines 119 *different* labs' experiments — different reactors, different light sources, different catalysts, even different units of reporting. It's less like one calibration curve and more like pooling 119 different calibration curves from 119 different instruments and asking one model to fit all of them at once. Given that, R² = 0.776 represents the model successfully learning real, generalizable chemistry trends despite substantial lab-to-lab variability — not a limitation of the modeling approach itself.

---


> 💡 **Why isn't R² closer to 1.0, like a good calibration curve?** A UV-Vis calibration curve is built from one instrument, one compound, and controlled standards — everything except concentration is held constant, so R² > 0.99 is expected. This dataset instead combines 119 *different* labs' experiments — different reactors, different light sources, different catalysts, even different units of reporting. It's less like one calibration curve and more like pooling 119 different calibration curves from 119 different instruments and asking one model to fit all of them at once. Given that, R² = 0.776 represents the model successfully learning real, generalizable chemistry trends despite substantial lab-to-lab variability — not a limitation of the modeling approach itself.

---


## 🗺️ How this notebook is organized

| Section | What happens | Chemistry analogy |
|---|---|---|
| 1. Setup | Install the ML libraries | Like installing a new plugin for your analysis software |
| 2. Load data | Read in the 909-experiment spreadsheet | Importing a spreadsheet of assay results |
| 3. Define features | Sort columns into "categorical" vs "numeric" | Sorting your DOE factors into qualitative (catalyst type) vs quantitative (pH, temp) |
| 4. Handle missing data | Flag which conditions weren't reported | Noting "N/A" vs an actual measured zero |
| 5. Engineer new features | Create useful combinations (e.g. power ÷ volume) | Calculating derived quantities like molarity from mass and volume |
| 6. Transform the target & split data | Log-transform yield; hold back 20% of data for testing | Log-transforming pH-like data; keeping held-out validation standards |
| 7. Encode catalyst names as numbers | Turn category names into predictive numbers | Building a lookup table of "average yield per catalyst" from historical data |
| 8. Build the 3 models + combine them | Train CatBoost, Random Forest, XGBoost; blend them | Getting 3 independent measurements and taking a weighted average |
| 9. Train final models | Retrain on all available training data | Final calibration run before reporting numbers |
| 10. Evaluate | Compute R², RMSE, MAE | Assessing your calibration curve's goodness-of-fit |
| 11. SHAP interpretation | Explain *why* the model predicts what it predicts | ANOVA-style "which factor mattered most" analysis |
| 12. Plots | Visualize everything | Making your results figures |

Every code cell below has a markdown cell right above it explaining, in plain language, what it does and why — you shouldn't need to know Python to follow along, though you will need to know it to *run* the notebook.

---


## 📖 Quick glossary (read this once, refer back as needed)

| ML term | What it means, in chemistry terms |
|---|---|
| **Feature** | An input variable — e.g. pH, catalyst loading, light source. Same idea as a factor in a DOE table. |
| **Target** | The thing you're trying to predict — here, H₂ production rate (μmol g⁻¹h⁻¹). The "response" in DOE terms. |
| **Model / regressor** | A mathematical function that takes features in and predicts the target out — like a fitted calibration equation, but far more flexible than a straight line. |
| **Training set / test set** | You fit the model on 80% of the data (training set) and check its predictions on the other 20% it never saw (test set) — analogous to calibrating on standards, then checking against an independent unknown. |
| **Decision tree** | A model that works like a flowchart: "Is catalyst loading > 0.15 g/L? → yes → is pH > 5? → ..." until it reaches a predicted yield. |
| **Random Forest** | Hundreds of decision trees, each trained on a random subset of the data, with predictions averaged — like running the same assay many times with slightly different reagent batches and averaging out the noise. |
| **Gradient boosting (CatBoost, XGBoost)** | Trees built one after another, where each new tree focuses on correcting the previous trees' mistakes — like iteratively refining a titration endpoint. |
| **Ensemble / stacking** | Combining multiple models' predictions into one, usually more accurate than any single model — like averaging replicate measurements from different instruments. |
| **R² (R-squared)** | Same R² you know from calibration curves: the fraction of variation in the target that the model explains. 1.0 = perfect fit, 0 = no better than just guessing the average every time. |
| **MSE / RMSE** | Mean Squared Error / its square root. RMSE is in the *same units as your measurement* (μmol g⁻¹h⁻¹ here), so it's directly interpretable as "typical prediction error," similar to a standard error of estimate. |
| **MAE** | Mean Absolute Error — the average size of the error, without the extra weight MSE puts on big outlier misses. |
| **Overfitting** | When a model memorizes the training data's quirks instead of learning general trends, so it performs great on training data but poorly on new data — like a calibration curve fit with a needlessly high-order polynomial through noisy points. |
| **Cross-validation (CV)** | Splitting the training data into several folds, training on some and validating on the rest, rotating through all folds — a more robust check than a single train/test split, similar to running replicate calibration checks. |
| **SHAP value** | For one specific prediction, how much each feature pushed the prediction up or down relative to the average — like decomposing a measured effect into contributions from each DOE factor. |
| **Hyperparameter** | A setting you choose *before* training (e.g. how many trees to grow) — not something the model learns itself, analogous to choosing your instrument's scan rate or integration time before a run. |

---


## 1. Setup

Colab does **not** come with `catboost`, `xgboost`, or `shap` pre-installed, so we install them first. Everything else (`pandas` for spreadsheets, `numpy` for numerical arrays, `scikit-learn` for the standard ML toolkit, `matplotlib` for plots) is already available in Colab.


In [ ]:
!pip install -q catboost xgboost shap

In [ ]:
import sys, time
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import shap
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor, Pool
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42  # fixes the "random" choices so results are reproducible run-to-run

_T0 = time.time()
def log(msg):
    # simple progress-timer print, so long-running cells still show life signs
    print(f"[{time.time()-_T0:7.1f}s] {msg}", flush=True)

print("Libraries loaded successfully.")

## 2. Load your data

Upload your Excel file directly in this Colab session (a file picker will pop up when you run the next cell), **or** mount Google Drive if the file already lives there. Use whichever is easier — only run one of the two options.

**What's in the file?** Each row is one experiment: a catalyst, its preparation method, the reaction conditions used, and the H₂ production rate that was measured (or reported in the source paper).


In [ ]:
# --- OPTION A: upload the file directly from your computer ---
from google.colab import files
uploaded = files.upload()          # choose your .xlsx file in the dialog
DATA_PATH = list(uploaded.keys())[0]
print("Using uploaded file:", DATA_PATH)

In [ ]:
# --- OPTION B: use a file already saved in Google Drive (skip Option A if using this) ---
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = "/content/drive/MyDrive/<path-to-your-file>.xlsx"
# print("Using Drive file:", DATA_PATH)

In [ ]:
df = pd.read_excel(DATA_PATH)

# The source file has a mangled '©' character in two column headers -> fix it
df = df.rename(columns={
    'Semiconductor Calcination TempÂ©': 'Semiconductor Calcination Temp(C)',
    'Photocatalyst Calcination Temp Â©': 'Photocatalyst Calcination Temp(C)',
})

TARGET = 'H2vproduction rate(micromol/g.h)'  # this is the column we're trying to predict

print("=" * 70)
print("  H2 PHOTOCATALYTIC PRODUCTION - CatBoost + RF + XGBoost + SHAP")
print("=" * 70)
print(f"  Dataset : {df.shape[0]} experiments, {df.shape[1]} columns")
print(f"  Target  : {df[TARGET].min():.0f} - {df[TARGET].max():.0f}  micromol/g.h")
df.head()  # peek at the first few rows, same idea as scrolling your raw data file

## 3. Define the features (your "DOE factors")

We split the columns into two types, the same distinction you'd make setting up a DOE table:

- **Categorical features** — qualitative factors with named levels, like catalyst identity or light source (e.g. "UV" vs "Visible"). There's no numeric order to these — "TiO₂" isn't "more" or "less" than "ZnO."
- **Numeric features** — quantitative factors you'd normally plot on a continuous axis: pH, temperature, loading (g/L), bandgap (eV), etc.

`Reference` (which paper/lab the experiment came from) is included as a categorical feature too — this matters a lot, as you'll see in the SHAP results, because it implicitly captures reactor design and measurement-protocol differences between labs that aren't otherwise recorded as separate columns.


In [ ]:
CAT_COLS = [
    'Cocatalyst 1', 'Semiconductor 1', 'Semiconductor 2', 'Semiconductor 3',
    'Form', 'Structure', 'Light', 'Light Source',
    'Preparation of Semiconductor', 'Preparation of Photocatalyst',
    'Reference',
]

NUM_COLS = [
    'Semiconductor 1 %', 'Semiconductor 2 %', 'Co-Catalyst wt',
    'Power (W)', 'Filter (nm)', 'Photocatalyst load (g/L)',
    'Semiconductor Calcination Temp(C)', 'Semiconductor Calcination Time (h)',
    'Photocatalyst Calcination Temp(C)', 'Photocatalyst Calcination Time (h)',
    'Glycerol Concentration (v%)', 'Solution Volume (ml)',
    'pH', 'Reaction Temp (C)', 'Bandgap (eV)',
]

print(f"Categorical (qualitative) factors: {len(CAT_COLS)}")
print(f"Numeric (quantitative) factors:    {len(NUM_COLS)}")

## 4. Handle missing data

Not every paper reports every condition. For example, if no optical filter was used, `Filter (nm)` is simply blank — that's different from a filter genuinely set to 0 nm. Before filling in these blanks, we create a **"was this reported?" flag** for each numeric column (1 = yes, 0 = no/blank).

**Why bother?** The *absence* of a reported filter is itself chemically meaningful — it usually means full-spectrum light reached the catalyst rather than a specific cut-on wavelength, which changes the accessible photon energy. If we just quietly filled in a default value without flagging it, the model would lose that information. It's the same reasoning as recording "not measured" separately from "measured as zero" in a lab notebook.


In [ ]:
for col in NUM_COLS:
    df[f'{col}_missing'] = df[col].isna().astype(int)
MISSING_FLAGS = [f'{c}_missing' for c in NUM_COLS]

for col in CAT_COLS:
    df[col] = df[col].fillna('None').astype(str)  # blank category -> explicit "None" label

print("Missing-value flags created:", len(MISSING_FLAGS))
print("Example - how many experiments didn't report a filter:", df['Filter (nm)_missing'].sum())

## 5. Engineer new features (derived quantities)

Just like you'd calculate molarity from mass and volume, or a rate constant from a slope, we compute new columns from combinations of the raw ones — because some chemically meaningful quantities are ratios or products of the raw inputs, not the raw inputs themselves. A few examples:

- `power_per_vol` = lamp power ÷ solution volume — a proxy for photon flux density delivered to the reaction mixture, not just raw lamp wattage.
- `load_x_glycerol` = catalyst loading × glycerol (hole-scavenger) concentration — captures the *interaction* between how much catalyst is present and how much sacrificial agent is available to consume photogenerated holes.
- `log_bandgap`, `log_power`, etc. — log-transformed versions of skewed numeric columns, for the same reason chemists often plot rate data on a log axis: it keeps a few very large values from dominating visually (and, here, from dominating the model's fitting process).

None of this changes the underlying chemistry — it just gives the model easier access to combinations that are already known (from photocatalysis literature) to matter.


In [ ]:
df['has_cocatalyst']     = (df['Cocatalyst 1']    != 'None').astype(int)
df['has_semiconductor2'] = (df['Semiconductor 2'] != 'None').astype(int)
df['has_semiconductor3'] = (df['Semiconductor 3'] != 'None').astype(int)
df['has_filter']         = df['Filter (nm)'].notna().astype(int)

df['power_per_vol']      = df['Power (W)'] / (df['Solution Volume (ml)'] + 1)
df['power_x_load']       = df['Power (W)'] * df['Photocatalyst load (g/L)']
df['log_power']          = np.log1p(df['Power (W)'])
df['log_vol']            = np.log1p(df['Solution Volume (ml)'])
df['log_load']           = np.log1p(df['Photocatalyst load (g/L)'])

df['load_x_glycerol']    = df['Photocatalyst load (g/L)'] * df['Glycerol Concentration (v%)']
df['glycerol_vol']       = df['Glycerol Concentration (v%)'] * df['Solution Volume (ml)']
df['wt_per_load']        = df['Co-Catalyst wt'] / (df['Photocatalyst load (g/L)'] + 1e-3)
df['wt_x_glycerol']      = df['Co-Catalyst wt'] * df['Glycerol Concentration (v%)']
df['log_glycerol']       = np.log1p(df['Glycerol Concentration (v%)'])
df['log_wt']             = np.log1p(df['Co-Catalyst wt'])
df['load_x_pH']          = df['Photocatalyst load (g/L)'] * df['pH']

df['sem_balance']        = df['Semiconductor 1 %'] - df['Semiconductor 2 %']
df['sem1_x_bg']          = df['Semiconductor 1 %'] * df['Bandgap (eV)']

df['log_bandgap']        = np.log1p(df['Bandgap (eV)'])
df['bandgap_pwr']        = df['Bandgap (eV)'] * df['Power (W)']
df['power_per_bg']       = df['Power (W)'] / (df['Bandgap (eV)'] + 1e-3)
df['log_cocatalyst_wt']  = np.log1p(df['Co-Catalyst wt'])

df['filter_x_power']     = df['Filter (nm)'] * df['Power (W)']
df['filter_x_bandgap']   = df['Filter (nm)'] * df['Bandgap (eV)']
df['log_filter']         = np.log1p(df['Filter (nm)'])

df['calc_prod']          = df['Semiconductor Calcination Temp(C)']  * df['Semiconductor Calcination Time (h)']
df['photo_calc']         = df['Photocatalyst Calcination Temp(C)']  * df['Photocatalyst Calcination Time (h)']

ENG_COLS = [
    c for c in df.columns
    if c not in CAT_COLS + NUM_COLS + MISSING_FLAGS + [TARGET, 'log_target', 'Exp']
]
FEATURES = CAT_COLS + NUM_COLS + MISSING_FLAGS + ENG_COLS

print(f"Total features: {len(FEATURES)}  "
      f"({len(CAT_COLS)} categorical + {len(NUM_COLS)} numeric + {len(MISSING_FLAGS)} missing-flags "
      f"+ {len(ENG_COLS)} engineered)")

## 6. Transform the target and split the data

**Why log-transform the target?** H₂ production rate in this dataset spans **0 to 269,120 μmol g⁻¹h⁻¹** — a huge range where most experiments cluster at the low end and a handful of exceptional catalysts produce far more. This is exactly the situation where chemists reach for a log axis (think Arrhenius plots, or plotting pH instead of raw [H⁺]): without it, the few huge values would dominate the fitting process and the model would essentially ignore the differences between typical, low-yield experiments. We apply `log(1 + yield)` before training and convert predictions back to normal units (`exp(prediction) - 1`) afterward, so every metric you see later is in real μmol g⁻¹h⁻¹ units.

**Why split into train/test sets?** We hold back 20% of the experiments (182 of them) that the model *never sees during training*, and only check its predictions against them at the very end. This is the machine-learning equivalent of validating a calibration curve against independent check standards rather than just checking how well it fits the points it was built from — fitting error on your own calibration standards will always look artificially good.

We also fill in missing numeric values using the **median computed only from the training set** (not the whole dataset) — using test-set information during training, even indirectly through a summary statistic, would be a form of look-ahead bias.


In [ ]:
df['log_target'] = np.log1p(df[TARGET])  # log(1 + yield): the "+1" avoids log(0) errors for zero-yield experiments

X      = df[FEATURES].copy()
y      = df['log_target'].copy()
y_orig = df[TARGET].copy()   # keep the original (non-log) values around for reporting real-unit metrics later

X_train, X_test, y_train, y_test, yo_train, yo_test = train_test_split(
    X, y, y_orig, test_size=0.2, random_state=RANDOM_STATE
)

train_medians = X_train[NUM_COLS].median()
X_train[NUM_COLS] = X_train[NUM_COLS].fillna(train_medians)
X_test[NUM_COLS]  = X_test[NUM_COLS].fillna(train_medians)  # same medians applied to test set - no peeking

eng_num = [c for c in ENG_COLS if X_train[c].dtype != object]
eng_medians = X_train[eng_num].median()
X_train[eng_num] = X_train[eng_num].fillna(eng_medians)
X_test[eng_num]  = X_test[eng_num].fillna(eng_medians)

cat_idx = [X_train.columns.get_loc(c) for c in CAT_COLS]
print(f"  Training experiments: {len(X_train)}  |  Held-out test experiments: {len(X_test)}")

## 7. Turn catalyst names into numbers (for Random Forest & XGBoost)

CatBoost can work with text labels like "Pt" or "TiO₂" directly. Random Forest and XGBoost can't — they need numbers. The naive approach (assign "TiO₂"=1, "ZnO"=2, "Pt"=3, ...) is a bad idea, because it invents a fake numeric order between category names that don't actually have one — the model might wrongly learn that "category 3" is "between" categories 2 and 4.

Instead we use **target encoding**: replace each catalyst name with the *average H₂ yield historically associated with that catalyst* — essentially building a lookup table from past screening data, the same way you might rank catalysts by their mean activity across previous campaigns before choosing which to test next.

**The catch, and how we handle it:** if we naively used the *whole* training set to build this lookup table, a catalyst's encoded value would partly be derived from its own yield, leaking the answer into the input. We prevent this the same way you'd design a blind study: for every group of experiments, their catalyst's "historical average" is computed using *only the other groups' data* (5-fold cross-validation), so no experiment's encoding is influenced by its own result.


In [ ]:
GLOBAL_MEAN = y_train.mean()
SMOOTHING = 8  # higher = trust the overall average more when a category has very few examples

def target_encode_column(train_col, train_target, other_col, n_splits=5, smoothing=SMOOTHING):
    encoded_train = pd.Series(index=train_col.index, dtype=float)
    kf_te = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    for tr_i, ho_i in kf_te.split(train_col):
        stats = train_target.iloc[tr_i].groupby(train_col.iloc[tr_i]).agg(['mean', 'count'])
        smoothed = (stats['mean'] * stats['count'] + GLOBAL_MEAN * smoothing) / (stats['count'] + smoothing)
        encoded_train.iloc[ho_i] = train_col.iloc[ho_i].map(smoothed).fillna(GLOBAL_MEAN).values

    full_stats = train_target.groupby(train_col).agg(['mean', 'count'])
    full_smoothed = (full_stats['mean'] * full_stats['count'] + GLOBAL_MEAN * smoothing) / (full_stats['count'] + smoothing)
    encoded_other = other_col.map(full_smoothed).fillna(GLOBAL_MEAN)
    return encoded_train.values, encoded_other.values

def build_target_encoded_features(train_df, target, other_df):
    tr_enc = train_df.copy()
    oth_enc = other_df.copy()
    for c in CAT_COLS:
        te_tr, te_oth = target_encode_column(train_df[c], target, other_df[c])
        tr_enc[c] = te_tr
        oth_enc[c] = te_oth
    return tr_enc.astype(float), oth_enc.astype(float)

X_train_enc, X_test_enc = build_target_encoded_features(X_train, y_train, X_test)
log("Target encoding built for RF/XGBoost (train+test)")

## 8. Build the three models

We use three different types of model, each with a slightly different "philosophy" for learning from the data — the same reason you might trust a result more if it's confirmed by three independent analytical techniques rather than one.

- **CatBoost** — builds decision trees (flowcharts: "if pH > 5, go this way; else go that way") one after another, where each new tree focuses on fixing the mistakes of the previous ones. It also handles catalyst names natively, so it gets the "cleanest" version of the categorical data.
- **Random Forest** — builds hundreds of decision trees independently, each on a random subset of the data and features, then averages their predictions. This tends to smooth out noise, similar to averaging many replicate measurements.
- **XGBoost** — conceptually similar to CatBoost (trees correcting each other's errors), but with a different internal growth strategy and regularization approach, so it tends to make different mistakes than CatBoost — which is exactly why it's useful to combine the two.

The settings below (`learning_rate`, `depth`, `n_estimators`, etc.) are **hyperparameters** — think of them like instrument settings you dial in *before* the run (scan rate, integration time), rather than something the model determines on its own. These particular values were tuned by testing several combinations and keeping the ones that generalized best to held-out data.


In [ ]:
CB_PARAMS = dict(
    learning_rate=0.025, depth=9, l2_leaf_reg=4,
    subsample=0.85, colsample_bylevel=0.75, min_data_in_leaf=2,
    grow_policy='Lossguide', max_leaves=200,
    loss_function='RMSE', random_seed=RANDOM_STATE, verbose=0,
)

RF_PARAMS = dict(
    n_estimators=500, max_depth=None, min_samples_leaf=2,
    max_features='sqrt', n_jobs=-1, random_state=RANDOM_STATE,
)

XGB_PARAMS = dict(
    n_estimators=2000, learning_rate=0.03, max_depth=6,
    subsample=0.85, colsample_bytree=0.75, min_child_weight=3,
    reg_lambda=1.0, reg_alpha=0.0, random_state=RANDOM_STATE,
    tree_method='hist', eval_metric='rmse',
    early_stopping_rounds=150,
)

N_FOLDS = 5

def fit_catboost(Xtr, ytr, Xval=None, yval=None, cat_features=None, n_iter=3000):
    m = CatBoostRegressor(iterations=n_iter, early_stopping_rounds=200, **CB_PARAMS)
    if Xval is not None:
        m.fit(Pool(Xtr, ytr, cat_features=cat_features),
              eval_set=Pool(Xval, yval, cat_features=cat_features))
    else:
        m.fit(Pool(Xtr, ytr, cat_features=cat_features))
    return m

def fit_rf(Xtr, ytr):
    m = RandomForestRegressor(**RF_PARAMS)
    m.fit(Xtr, ytr)
    return m

def fit_xgb(Xtr, ytr, Xval, yval):
    m = XGBRegressor(**XGB_PARAMS)
    m.fit(Xtr, ytr, eval_set=[(Xval, yval)], verbose=False)
    return m

print("Model builders defined.")

## 9. Combine the three models into one ("stacking")

Rather than just averaging the three models equally, we let the data tell us the best *weighted* combination. Here's how, step by step:

1. Split the training data into 5 folds (groups). For each fold, train all three models on the other 4 folds and predict on the held-out fold. Repeat until every training experiment has been predicted by all three models *without that experiment being used to train the model that predicted it* — this is called **cross-validation**, and it's the same logic as a blind proficiency test.
2. Feed those "honest" predictions into a simple linear model (**Ridge regression**) that learns how much weight to give each of the three models' predictions. If one model is consistently more reliable, it earns a higher weight — automatically, from the data, rather than us guessing.

⚠️ **This cell is the slow one** — training 3 models × 5 folds takes a while on Colab's free CPU runtime (expect tens of minutes). If you want it faster, switch to a GPU runtime (`Runtime → Change runtime type → T4 GPU`); CatBoost and XGBoost both support GPU training with minor parameter changes (`task_type='GPU'` for CatBoost, `device='cuda'` for XGBoost).


In [ ]:
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

oof_cb  = np.zeros(len(X_train))
oof_rf  = np.zeros(len(X_train))
oof_xgb = np.zeros(len(X_train))
cb_best_iters = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train), 1):
    ci = cat_idx
    log(f"Fold {fold}/{N_FOLDS} starting...")

    Xtr_cb, Xval_cb = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    ytr, yval = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    m_cb = fit_catboost(Xtr_cb, ytr, Xval_cb, yval, cat_features=ci)
    oof_cb[val_idx] = m_cb.predict(Pool(Xval_cb, cat_features=ci))
    cb_best_iters.append(m_cb.best_iteration_)
    log(f"Fold {fold}: CatBoost done (best_iter={m_cb.best_iteration_})")

    Xtr_rf, Xval_rf = build_target_encoded_features(
        X_train.iloc[tr_idx], y_train.iloc[tr_idx], X_train.iloc[val_idx]
    )

    m_rf = fit_rf(Xtr_rf, ytr)
    oof_rf[val_idx] = m_rf.predict(Xval_rf)
    log(f"Fold {fold}: RandomForest done")

    m_xgb = fit_xgb(Xtr_rf, ytr, Xval_rf, yval)
    oof_xgb[val_idx] = m_xgb.predict(Xval_rf)
    log(f"Fold {fold}: XGBoost done")

    fold_r2_cb  = r2_score(np.expm1(yval), np.expm1(oof_cb[val_idx]))
    fold_r2_rf  = r2_score(np.expm1(yval), np.expm1(oof_rf[val_idx]))
    fold_r2_xgb = r2_score(np.expm1(yval), np.expm1(oof_xgb[val_idx]))
    log(f"Fold {fold}: CatBoost R2={fold_r2_cb:.4f}  RF R2={fold_r2_rf:.4f}  XGB R2={fold_r2_xgb:.4f}")

best_n_cb = int(np.mean(cb_best_iters))

meta_X = np.column_stack([oof_cb, oof_rf, oof_xgb])
meta = Ridge(alpha=1.0, positive=True)  # positive=True keeps weights non-negative and interpretable
meta.fit(meta_X, y_train)
print(f"\n  Meta-model weights  ->  CatBoost: {meta.coef_[0]:.3f}  "
      f"RF: {meta.coef_[1]:.3f}  XGBoost: {meta.coef_[2]:.3f}  (intercept {meta.intercept_:.3f})")

> **Reading the weights:** in our run, CatBoost got the highest weight (~0.87), XGBoost a smaller supporting weight (~0.16), and Random Forest essentially zero. That doesn't mean Random Forest is "bad" — it means it wasn't telling the combination anything that CatBoost and XGBoost weren't already capturing between them, similar to how a third replicate measurement adds little if the first two already agree closely with each other.


## 10. Train the final models

The cross-validation loop above was only for *learning the combination weights*. Now we retrain each model one more time using **all** of the training data (not just 4/5 of it per fold), since more training data generally makes a model better, and we've already learned how to blend them.


In [ ]:
final_cb = CatBoostRegressor(iterations=best_n_cb, **CB_PARAMS)
final_cb.fit(Pool(X_train, y_train, cat_features=cat_idx))
log("Final CatBoost trained")

final_rf = fit_rf(X_train_enc, y_train)
log("Final RandomForest trained")

Xtr_f, Xval_f, ytr_f, yval_f = train_test_split(
    X_train_enc, y_train, test_size=0.12, random_state=RANDOM_STATE
)
final_xgb = fit_xgb(Xtr_f, ytr_f, Xval_f, yval_f)
log("Final XGBoost trained")

## 11. Evaluate the model on data it has never seen

Now we check predictions against the **182 held-out test experiments** that no model has trained on. Four metrics are reported, all standard in analytical method validation:

- **R²** — same as your calibration curve R²: fraction of variance explained (0 to 1, higher is better).
- **MSE (Mean Squared Error)** — average of the squared prediction errors. Squaring means a few very large misses dominate this number, similar to how outliers heavily influence a sum-of-squares residual.
- **RMSE (Root Mean Squared Error)** — square root of MSE, which brings it back into the same units as your measurement (μmol g⁻¹h⁻¹), so you can read it as "typical error size."
- **MAE (Mean Absolute Error)** — average error size without the extra weight on big misses; a more "typical case" error estimate than RMSE.


In [ ]:
def metrics(y_true, y_pred, label):
    r2   = r2_score(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred)
    print(f"  {label:<22} R2={r2:.4f}   MSE={mse:,.1f}   RMSE={rmse:,.1f}   MAE={mae:,.1f}")
    return dict(label=label, r2=r2, mse=mse, rmse=rmse, mae=mae)

pred_test_cb_log  = final_cb.predict(Pool(X_test, cat_features=cat_idx))
pred_test_rf_log  = final_rf.predict(X_test_enc)
pred_test_xgb_log = final_xgb.predict(X_test_enc)

pred_test_cb  = np.expm1(pred_test_cb_log)   # convert back from log-space to real mu mol/g.h units
pred_test_rf  = np.expm1(pred_test_rf_log)
pred_test_xgb = np.expm1(pred_test_xgb_log)

meta_test_X   = np.column_stack([pred_test_cb_log, pred_test_rf_log, pred_test_xgb_log])
pred_test_ens_log = meta.predict(meta_test_X)
pred_test_ens = np.expm1(pred_test_ens_log)

pred_test_avg = np.expm1((pred_test_cb_log + pred_test_rf_log + pred_test_xgb_log) / 3)

print("=" * 70)
print("  TEST SET RESULTS")
print("=" * 70)
res = []
res.append(metrics(yo_test, pred_test_cb,  "CatBoost"))
res.append(metrics(yo_test, pred_test_rf,  "RandomForest"))
res.append(metrics(yo_test, pred_test_xgb, "XGBoost"))
res.append(metrics(yo_test, pred_test_avg, "Simple average"))
res.append(metrics(yo_test, pred_test_ens, "Stacked ensemble (Ridge)"))

best = max(res, key=lambda d: d['r2'])
print(f"\n  Best model on test set: {best['label']}  (R2={best['r2']:.4f}, MSE={best['mse']:,.1f})")

In [ ]:
# Train-set sanity check: is the model overfitting?
# (If train R2 were dramatically higher than test R2, that would signal overfitting -
#  the model memorizing training quirks instead of learning general chemistry trends.)
pred_train_cb_log  = final_cb.predict(Pool(X_train, cat_features=cat_idx))
pred_train_rf_log  = final_rf.predict(X_train_enc)
pred_train_xgb_log = final_xgb.predict(X_train_enc)
meta_train_X = np.column_stack([pred_train_cb_log, pred_train_rf_log, pred_train_xgb_log])
pred_train_ens = np.expm1(meta.predict(meta_train_X))
print("TRAIN SET (sanity check for overfitting):")
metrics(yo_train, pred_train_ens, "Stacked ensemble (Ridge)")

## 12. SHAP: which factors actually drive H₂ yield?

Knowing the model predicts well is useful, but as a chemist you'll want to know *why* it makes the predictions it does — the equivalent of an effects analysis after a DOE campaign. **SHAP (SHapley Additive exPlanations)** answers this for each individual prediction: for one specific experiment, it tells you how much each feature (catalyst type, pH, loading, etc.) pushed the predicted yield up or down relative to the "average" experiment.

Averaged across all 182 test experiments, this ranks which features matter most overall — that's what the beeswarm and bar plots below show. Each dot in the beeswarm plot is one experiment; its color shows whether that feature's value was high (red) or low (blue) for that experiment, and its horizontal position shows whether that pushed the prediction up or down.


In [ ]:
explainer  = shap.TreeExplainer(final_cb)
shap_test  = explainer.shap_values(Pool(X_test, cat_features=cat_idx))
base_value = explainer.expected_value

mean_abs_shap = np.abs(shap_test).mean(axis=0)
feat_rank     = np.argsort(mean_abs_shap)[::-1]
top_numeric   = [FEATURES[i] for i in feat_rank if FEATURES[i] in NUM_COLS + ENG_COLS][:4]

print("Top 15 features by mean |SHAP| (i.e. average influence on predicted yield):")
for i in feat_rank[:15]:
    mean_shap = shap_test[:, i].mean()
    direction = "pushes yield UP on average" if mean_shap > 0 else "pushes yield DOWN on average"
    print(f"    {FEATURES[i]:<42} {mean_abs_shap[i]:>10.4f}  ({direction})")

In [ ]:
plt.figure(figsize=(12, 9))
shap.summary_plot(shap_test, X_test, feature_names=FEATURES, max_display=20, show=False)
plt.title("SHAP Beeswarm - Feature Impact on H2 Prediction (Test Set)", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_summary_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(11, 8))
shap.summary_plot(shap_test, X_test, feature_names=FEATURES, plot_type='bar', max_display=20, show=False)
plt.title("SHAP Bar Chart - Mean |SHAP| Importance (Test Set)", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_summary_bar.png', dpi=150, bbox_inches='tight')
plt.show()

**Dependence plots** below zoom in on the top 4 *numeric* features and show the relationship between each feature's actual value (x-axis) and its effect on predicted yield (y-axis) — similar to plotting a response surface slice from a DOE model, one factor at a time.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()
for ax_i, feat in enumerate(top_numeric):
    feat_idx = FEATURES.index(feat)
    sc = axes[ax_i].scatter(X_test[feat].values, shap_test[:, feat_idx],
                             c=X_test[feat].values, cmap='coolwarm', alpha=0.65, s=28)
    axes[ax_i].axhline(0, color='black', linestyle='--', lw=1, alpha=0.5)
    axes[ax_i].set_xlabel(feat); axes[ax_i].set_ylabel('SHAP value (log H2)')
    axes[ax_i].set_title(f'Dependence: {feat}', fontweight='bold')
    axes[ax_i].grid(True, linestyle='--', alpha=0.35)
    plt.colorbar(sc, ax=axes[ax_i], shrink=0.85)
fig.suptitle("SHAP Dependence Plots - Top 4 Numeric Features", fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('shap_dependence_grid.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Parity plot and model comparison

A **parity plot** plots predicted vs. actual yield for every experiment — the closer the points sit to the diagonal (y = x) line, the better the predictions. This is the ML equivalent of a "measured vs. expected" plot you'd make to sanity-check a calibration.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
ax = axes[0]
ax.scatter(yo_train, pred_train_ens, alpha=0.35, s=18, color='steelblue')
lim = max(yo_train.max(), pred_train_ens.max()) * 1.05
ax.plot([0, lim], [0, lim], 'r--', lw=1.5)
r2_tr = r2_score(yo_train, pred_train_ens)
ax.set_title(f'Train (ensemble) | R2 = {r2_tr:.4f}', fontsize=13, fontweight='bold')
ax.set_xlabel('Observed H2 (micromol/g.h)'); ax.set_ylabel('Predicted H2 (micromol/g.h)')
ax.grid(True, linestyle='--', alpha=0.4)

ax = axes[1]
ax.scatter(yo_test, pred_test_ens, alpha=0.5, s=22, color='darkorange')
lim = max(yo_test.max(), pred_test_ens.max()) * 1.05
ax.plot([0, lim], [0, lim], 'r--', lw=1.5)
ens_row = [r for r in res if r['label'] == 'Stacked ensemble (Ridge)'][0]
ax.set_title(f"Test (ensemble) | R2 = {ens_row['r2']:.4f}  MSE = {ens_row['mse']:,.0f}",
             fontsize=13, fontweight='bold')
ax.set_xlabel('Observed H2 (micromol/g.h)'); ax.set_ylabel('Predicted H2 (micromol/g.h)')
ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle('Stacked Ensemble - H2 Photocatalytic Production Rate Prediction', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('parity_plot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
labels = [r['label'] for r in res]
r2s    = [r['r2']  for r in res]
mses   = [r['mse'] for r in res]

axes[0].bar(labels, r2s, color=['#4575b4']*3 + ['#999999', '#d73027'])
axes[0].set_ylabel('R2 (test set)'); axes[0].set_title('R2 by model', fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

axes[1].bar(labels, mses, color=['#4575b4']*3 + ['#999999', '#d73027'])
axes[1].set_ylabel('MSE (test set)'); axes[1].set_title('MSE by model (lower = better)', fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)
axes[1].grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Final summary


In [ ]:
print("=" * 70)
print("  FINAL SUMMARY")
print("=" * 70)
for r in res:
    print(f"  {r['label']:<26} R2={r['r2']:.4f}  MSE={r['mse']:,.1f}  RMSE={r['rmse']:,.1f}  MAE={r['mae']:,.1f}")

## 15. Download the generated figures (optional)

Run this cell to download the five PNG files this notebook produced, e.g. to add them to a GitHub repo README or a slide deck.


In [ ]:
from google.colab import files as colab_files
for fname in ['shap_summary_beeswarm.png', 'shap_summary_bar.png',
              'shap_dependence_grid.png', 'parity_plot.png', 'model_comparison.png']:
    colab_files.download(fname)

---

## Where to go from here

- **Try a new catalyst hypothesis**: build a single-row DataFrame with your proposed conditions (matching the `FEATURES` columns), run it through `final_cb.predict()`, and see what yield the model expects — a quick sanity check before committing bench time.
- **Watch out for the `Reference` feature**: since it's the single strongest predictor, be cautious about trusting predictions for catalyst/condition combinations far outside what's been published before — the model is interpolating within known literature space, not extrapolating new chemistry.
- **A note on absolute error size**: MAE/RMSE look large in absolute terms (thousands of μmol g⁻¹h⁻¹) because the target itself spans 5 orders of magnitude across 119 different labs' worth of protocols — R² is the fairer relative metric for judging fit quality here.

## References

1. Bakır, R., Orak, C., Yüksel, A. (2024). Optimizing hydrogen evolution prediction: A unified approach using random forests, lightGBM, and Bagging Regressor ensemble model. *International Journal of Hydrogen Energy*, 67, 101–110. https://doi.org/10.1016/j.ijhydene.2024.04.173
2. Suriyaprakash, J., Saudagar, A.K.J., Wu, L., Shan, L. (2025). Machine learning-powered nanoengineering of flexible, durable and scalable photoelectrode for efficient H₂ production. *International Journal of Hydrogen Energy*, 189, 152214. https://doi.org/10.1016/j.ijhydene.2025.152214
